# 提示词注入和PVE防护

间接的提示词注入是典型的agent安全问题。攻击者将指令植入agent能够检索到的数据中，在投喂阶段，这些指令覆盖了用户提示词。在工具调用层面，将所有检索到的内容视为可以被执行的代码。

## 问题描述

LLMs 不能够可靠分辨出指令是来自用户还是被检索到的内容。一个PDF、网页、备忘录甚至上一轮agent会话可能携带攻击信息，然后模型会像用户要求了一样去执行它。

这就是2024-2026年最典型的agent安全问题。每个生产agent都需要对抗它。

## 基本概念

### 间接提示词注入

- 数据小偷。   agent泄漏会话历史给攻击者控制的URL。
- 蠕虫。  注入的上下文让agent把攻击放到下一轮输出中。
- 持久记忆投毒。  agent存储了攻击者的指令，在下轮会话的时候又中毒了。
- 信息生产污染。  被注入的事实通过共享记忆传播到了其他agents。
- 工具滥用。  注册表上的任何工具攻击者都能摸到。

关键声明：在工具调用层面，将所有检索到的提示词当成任意的代码执行。

### 防御信条

- 不信任任何检索到的内容。
- 黑/白名单。
- 每步安全评估。
- 工具输入和输出加护栏。
- 请求人确认。
- 外部存储捕捉上下文。

### PVE： Prompt-Validator-Exector

- 使用一个便宜、快速的验证模型在每次工具调用前做验证。
- 验证这些内容： 是否符合用户意图？是否触碰敏感内容？是否存在类似注入的内容？
- 如果验证器拒绝，主模型吐出错误信息，让换一条路。

权衡： 每次工具调用额外多了一次推理。对于大多数的生产级agent，这份安全很便宜。

### 哪些地方导致防御失效

- 上下文内容没有源信息。 模型没办法根据用户指令或者外部指令来区分许可证等级。
- 所有的护栏都在末尾。  模型在吐出最终答案前就已经触碰到了真实世界数据。
- 只靠指令遵循。  系统提示词说忽略所有没被信任的提示词并不是强制的。
- 记忆召回过度信任。  agent可能召回之前其他被投毒agents写到记忆里的内容。